# QFF 2026 PS-Q1 — Biomedical Signal Classifier | Variational Quantum Classification for Arrhythmia & Biosignals

This notebook uses PS-Q1 200-sample synthetic dataset.

The A/B comparison compiles the logical circuit for each processor and measures circuit resources; it does not claim noisy A/B classification accuracy.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.problem import make_dataset
from src.vqc import train_vqc, predict, bind_circuit_for_sample
from src.processors import load_processor_config
from src.routing import transpile_for_processor
from src.metrics import circuit_metrics, save_metrics_csv
from src.visualization import plot_coupling_graph, plot_training_loss, plot_depth_comparison

ROOT = Path.cwd()
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
(ROOT / "results/figures").mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT.resolve())

Project root: C:\Users\SASIDHAR\Downloads\QFF-2026-PS-Q1-main\QFF-2026-PS-Q1-main


## 1. Generate the specified dataset

In [2]:
X_train, X_test, y_train, y_test, scaler = make_dataset(random_state=42)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Train class counts:", np.bincount(y_train), "| Test class counts:", np.bincount(y_test))

Train shape: (160, 4) | Test shape: (40, 4)
Train class counts: [80 80] | Test class counts: [20 20]


## 2. Train and evaluate the VQC

In [3]:
model = train_vqc(X_train, y_train, maxiter=35, seed=42)
y_pred = predict(model, X_test)
classification = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "final_training_loss": float(model["final_loss"]),
    "optimizer_success": model["optimizer_success"],
    "optimizer_message": model["optimizer_message"],
}
classification

{'accuracy': 0.775,
 'precision': 0.8666666666666667,
 'recall': 0.65,
 'f1': 0.7428571428571429,
 'final_training_loss': 0.5401825521397494,
 'optimizer_success': True,
 'optimizer_message': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL'}

In [4]:
plot_training_loss(model["loss_history"], ROOT / "results/figures/training_loss.png")
print("Saved training loss plot.")

Saved training loss plot.


## 2B. Classical baseline (compatibility check)

In [5]:
# A classical baseline is useful for judging whether the VQC adds value.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline = LogisticRegression(random_state=42, max_iter=1000)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
classical_metrics = {
    "accuracy": float(accuracy_score(y_test, baseline_pred)),
    "precision": float(precision_score(y_test, baseline_pred, zero_division=0)),
    "recall": float(recall_score(y_test, baseline_pred, zero_division=0)),
    "f1": float(f1_score(y_test, baseline_pred, zero_division=0)),
}
print("VQC metrics:", classification)
print("Logistic-regression baseline:", classical_metrics)

VQC metrics: {'accuracy': 0.775, 'precision': 0.8666666666666667, 'recall': 0.65, 'f1': 0.7428571428571429, 'final_training_loss': 0.5401825521397494, 'optimizer_success': True, 'optimizer_message': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL'}
Logistic-regression baseline: {'accuracy': 0.875, 'precision': 0.8947368421052632, 'recall': 0.85, 'f1': 0.8717948717948718}


## 3. Load Processor A and B and visualize their coupling graphs

In [6]:
processor_a = load_processor_config(ROOT / "processors/processor_A.json")
processor_b = load_processor_config(ROOT / "processors/processor_B.json")
for config in [processor_a, processor_b]:
    print(config["name"], "| qubits:", config["num_qubits"], "| edges:", config["coupling_map"])
    safe_name = config["name"].replace(" ", "_")
    plot_coupling_graph(config["graph"], config["name"] + " coupling map",
        ROOT / f"results/figures/{safe_name}_coupling.png")

Processor A | qubits: 5 | edges: [[0, 1], [1, 2], [1, 3], [3, 4]]
Processor B | qubits: 9 | edges: [[0, 1], [1, 2], [1, 6], [2, 3], [2, 6], [2, 5], [3, 4], [4, 5], [5, 6], [6, 7], [6, 8], [8, 0]]


## 4. Compile the same trained logical circuit on both processors

In [7]:
# Bind the same held-out sample and trained weights before compiling.
logical = bind_circuit_for_sample(model, X_test[0])
compiled_a = transpile_for_processor(logical, processor_a)
compiled_b = transpile_for_processor(logical, processor_b)
rows = [
    circuit_metrics(compiled_a, processor_a["name"], processor_a["num_qubits"]),
    circuit_metrics(compiled_b, processor_b["name"], processor_b["num_qubits"]),
]
comparison = pd.DataFrame(rows)
comparison

,processor,logical_qubits,physical_qubits,compiled_qubits,depth,total_gates,cx_count,swap_count,operation_counts
0,Processor A,4,5,5,13,38,6,0,"{'sx': 16, 'rz': 16, 'cx': 6}"
1,Processor B,4,9,9,13,38,6,0,"{'sx': 16, 'rz': 16, 'cx': 6}"


In [8]:
save_metrics_csv(rows, ROOT / "results/tables/processor_comparison.csv")
plot_depth_comparison(rows, ROOT / "results/figures/circuit_depth_comparison.png")
summary = {
    "problem_statement": "PS-Q1 VQC synthetic-data variant",
    "dataset": {"samples": 200, "features": 4, "informative": 3, "random_state": 42, "train_samples": 160, "test_samples": 40},
    "classification_metrics": classification,
    "classical_baseline_metrics": classical_metrics,
    "processor_comparison": rows,
    "transpiler": {"optimization_level": 1, "seed": 42},
    "note": "Ideal statevector training; compiled resource comparison only; no noisy A/B accuracy claimed."
}
with open(ROOT / "results/tables/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Saved result tables and figures.")

Saved result tables and figures.
